<a href="https://colab.research.google.com/github/Yash1014-code/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yash1014-code/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule:

Prioritize pages that are old and still have meaningful search visibility, because these pages may benefit from a content refresh.

Reason codes:

* stale_but_visible — page is old and has meaningful search visibility
* stale_low_visibility — page is old but has low search visibility
* fresh_but_visible — page is relatively fresh but has meaningful visibility
* low_priority — page is neither sufficiently old nor sufficiently visible

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import pandas as pd
import os

# Download the FlyRank starter dataset from the GitHub repository
url = "https://raw.githubusercontent.com/Yash1014-code/flyrank-internship-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:
print(df.columns.tolist())


['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [6]:
# Signal 1: Content age
age_bins = [-1, 90, 180, 365, float("inf")]
age_labels = ["0-90 days", "91-180 days", "181-365 days", "365+ days"]

df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=age_bins,
    labels=age_labels
)

age_check = (
    df.groupby("age_bucket", observed=False)
      .agg(n=("content_id", "size"),
           avg_impressions=("impressions_90d", "mean"))
      .reset_index()
)

print(age_check)

     age_bucket      n  avg_impressions
0     0-90 days    492      3209.445122
1   91-180 days  11780      5101.726401
2  181-365 days  11368      5398.772871
3     365+ days   6360      5182.445755


In [7]:
# Signal 2: Search visibility
impression_bins = [-1, 100, 500, 1000, 5000, float("inf")]
impression_labels = [
    "0-100",
    "101-500",
    "501-1000",
    "1001-5000",
    "5000+"
]

df["impression_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=impression_bins,
    labels=impression_labels
)

impression_check = (
    df.groupby("impression_bucket", observed=False)
      .agg(n=("content_id", "size"),
           avg_age_days=("content_age_days", "mean"))
      .reset_index()
)

print(impression_check)

  impression_bucket     n  avg_age_days
0             0-100  8006    242.461029
1           101-500  5279    268.212730
2          501-1000  3206    267.696195
3         1001-5000  7359    262.383884
4             5000+  6150    250.224228


In [8]:
# Baseline rule:
# Prioritize pages that are old (>= 180 days)
# and still have meaningful search visibility (>= 500 impressions).

stale = df["content_age_days"] >= 180
visible = df["impressions_90d"] >= 500

# Score:
# Higher impressions = higher priority among stale and visible pages.
df["score"] = (
    df["impressions_90d"]
    .where(stale & visible, 0)
)

# Reason codes
df["reason_code"] = "low_priority"

df.loc[stale & ~visible, "reason_code"] = "stale_low_visibility"
df.loc[~stale & visible, "reason_code"] = "fresh_but_visible"
df.loc[stale & visible, "reason_code"] = "stale_but_visible"

# Action
df["action"] = "monitor"
df.loc[stale & visible, "action"] = "refresh"

# Rank from highest score to lowest
df = df.sort_values(
    by="score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Write the ranked queue
output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(df))

# Show top 10
df[
    ["rank", "content_id", "content_age_days",
     "impressions_90d", "score", "reason_code", "action"]
].head(10)

Saved: work/outputs/baseline_action_score.csv
Rows: 30000


,rank,content_id,content_age_days,impressions_90d,score,reason_code,action
0,1,content_5fe46e04994d,537,517715,517715,stale_but_visible,refresh
1,2,content_aaef01a50def,445,517109,517109,stale_but_visible,refresh
2,3,content_8c19996aa890,445,509252,509252,stale_but_visible,refresh
3,4,content_4c36c775b818,445,463103,463103,stale_but_visible,refresh
4,5,content_2dba2b1f9536,299,443434,443434,stale_but_visible,refresh
5,6,content_1a9e894be2e2,482,416180,416180,stale_but_visible,refresh
6,7,content_2c2606c5d176,362,347399,347399,stale_but_visible,refresh
7,8,content_db5989a78dd3,445,345111,345111,stale_but_visible,refresh
8,9,content_9532f197bbc8,445,309192,309192,stale_but_visible,refresh
9,10,content_8e7ba84a972b,224,288426,288426,stale_but_visible,refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The baseline selects pages that are at least 180 days old and have at least 500 impressions in the last 90 days. All top-20 pages meet both conditions, so they receive the `stale_but_visible` reason code and the `refresh` action. Confidence is based on the observed age and visibility signals only; the rule does not prove that refreshing a page will improve performance.

1. **Rank 1 — `content_5fe46e04994d`**: Refresh — 537 days old with 517,715 impressions, so it is highly visible and stale. **Confidence: High.** It could be wrong if the content is still accurate and does not need updating.

2. **Rank 2 — `content_aaef01a50def`**: Refresh — 445 days old with 517,109 impressions, making it both stale and highly visible. **Confidence: High.** It could be wrong if the page is already current despite its age.

3. **Rank 3 — `content_8c19996aa890`**: Refresh — 445 days old with 509,252 impressions, giving it strong visibility while being stale. **Confidence: High.** It could be wrong if the high impressions do not represent a real refresh opportunity.

4. **Rank 4 — `content_4c36c775b818`**: Refresh — 445 days old with 463,103 impressions, so it is old and has substantial visibility. **Confidence: High.** It could be wrong if the page's information is still up to date.

5. **Rank 5 — `content_2dba2b1f9536`**: Refresh — 299 days old with 443,434 impressions, indicating strong visibility from an older page. **Confidence: High.** It could be wrong if no meaningful content changes are needed.

6. **Rank 6 — `content_1a9e894be2e2`**: Refresh — 482 days old with 416,180 impressions, making it both old and highly visible. **Confidence: High.** It could be wrong if its existing content is still performing appropriately.

7. **Rank 7 — `content_2c2606c5d176`**: Refresh — 362 days old with 347,399 impressions, showing substantial visibility despite being nearly a year old. **Confidence: High.** It could be wrong if the page remains relevant without an update.

8. **Rank 8 — `content_db5989a78dd3`**: Refresh — 445 days old with 345,111 impressions, meeting both baseline conditions comfortably. **Confidence: High.** It could be wrong if age alone does not indicate that the content needs work.

9. **Rank 9 — `content_9532f197bbc8`**: Refresh — 445 days old with 309,192 impressions, giving it meaningful visibility while being stale. **Confidence: High.** It could be wrong if the page has no clear content-refresh opportunity.

10. **Rank 10 — `content_8e7ba84a972b`**: Refresh — 224 days old with 288,426 impressions, so it is relatively recently aged but still passes the six-month threshold and has strong visibility. **Confidence: High.** It could be wrong if 224 days is not long enough to justify a refresh.

11. **Rank 11 — `content_8451fc6f034d`**: Refresh — 280 days old with 272,144 impressions, satisfying both conditions. **Confidence: High.** It could be wrong if the content remains accurate and useful.

12. **Rank 12 — `content_66b4046cc144`**: Refresh — 225 days old with 217,415 impressions, so it is older than the threshold and still visible. **Confidence: High.** It could be wrong if the page does not need substantive changes.

13. **Rank 13 — `content_c8e9d6ab9013`**: Refresh — 362 days old with 208,678 impressions, indicating meaningful visibility from an older page. **Confidence: High.** It could be wrong if the page is already well maintained.

14. **Rank 14 — `content_b511d4bc4ad2`**: Refresh — 216 days old with 205,915 impressions, passing both the age and visibility thresholds. **Confidence: High.** It could be wrong if the six-month cutoff is too aggressive.

15. **Rank 15 — `content_d17681677e69`**: Refresh — 313 days old with 201,584 impressions, making it an older page with meaningful visibility. **Confidence: High.** It could be wrong if the page has no identifiable information that needs updating.

16. **Rank 16 — `content_a7427266c305`**: Refresh — 257 days old with 201,111 impressions, satisfying the baseline rule. **Confidence: High.** It could be wrong if the page's current content is already adequate.

17. **Rank 17 — `content_9463d30d5826`**: Refresh — 480 days old with 192,478 impressions, making it clearly stale while retaining meaningful visibility. **Confidence: High.** It could be wrong if the page's age does not correspond to outdated information.

18. **Rank 18 — `content_c5063073d048`**: Refresh — 211 days old with 192,205 impressions, so it passes both baseline thresholds. **Confidence: High.** It could be wrong if 211 days is insufficient evidence that a refresh is needed.

19. **Rank 19 — `content_01908772c6db`**: Refresh — 313 days old with 187,893 impressions, showing meaningful visibility from an older page. **Confidence: High.** It could be wrong if refreshing the page would not address any actual performance issue.

20. **Rank 20 — `content_f02b48f88241`**: Refresh — 223 days old with 181,514 impressions, meeting the baseline's age and visibility conditions. **Confidence: High.** It could be wrong if the page is still current and useful despite its age.

**Overall observation:** The top-20 list is heavily concentrated on pages with high impressions because impressions determine the score after the age and visibility conditions are met. This makes the baseline transparent, but it also means the rule may over-prioritize high-traffic pages and miss lower-traffic pages with a stronger need for refresh.


In [9]:
# Show the top 20 pages for manual review

top20 = df.head(20)[
    [
        "rank",
        "content_id",
        "content_age_days",
        "impressions_90d",
        "score",
        "reason_code",
        "action"
    ]
]

print(top20.to_string(index=False))


 rank           content_id  content_age_days  impressions_90d  score       reason_code  action
    1 content_5fe46e04994d               537           517715 517715 stale_but_visible refresh
    2 content_aaef01a50def               445           517109 517109 stale_but_visible refresh
    3 content_8c19996aa890               445           509252 509252 stale_but_visible refresh
    4 content_4c36c775b818               445           463103 463103 stale_but_visible refresh
    5 content_2dba2b1f9536               299           443434 443434 stale_but_visible refresh
    6 content_1a9e894be2e2               482           416180 416180 stale_but_visible refresh
    7 content_2c2606c5d176               362           347399 347399 stale_but_visible refresh
    8 content_db5989a78dd3               445           345111 345111 stale_but_visible refresh
    9 content_9532f197bbc8               445           309192 309192 stale_but_visible refresh
   10 content_8e7ba84a972b               224      

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks + leakage check

**Weak picks:**
The baseline may over-prioritize pages with very high impressions. A page can have high visibility and be old but still have accurate, useful, and up-to-date content. The rule also does not consider whether the page's performance is actually declining or whether there is a specific content problem. Therefore, some `refresh` recommendations may be false positives.

**Leakage check:**
The baseline uses only `content_age_days` and `impressions_90d`, which are observable input signals. It does not use `trend_direction`, `trend_pct`, or `is_declining_label`. It also does not use FlyRank product flags such as `health_score`, `needs_ctr_fix`, `is_quick_win`, `priority_score`, or `action_type`. No future-window data is used in the rule.

**Conclusion:**
This baseline is a transparent decision-support rule, not proof that a page needs refreshing. Its main weakness is that high-impression pages can dominate the ranking even when their actual content quality or freshness may not require an update.


In [10]:
# Leakage check: verify that our baseline uses only allowed input signals

baseline_features = ["content_age_days", "impressions_90d"]

print("Baseline features:", baseline_features)

# Confirm the columns exist
print("\nRequired columns present:")
for col in baseline_features:
    print(f"{col}: {col in df.columns}")

# Confirm prohibited label/trend fields are not used
prohibited = ["trend_direction", "trend_pct", "is_declining_label"]

print("\nProhibited fields used in baseline:")
for col in prohibited:
    print(f"{col}: {col in baseline_features}")

# Show the weak-pick pattern
print("\nTop 20 score range:")
print("Highest score:", df["score"].head(20).max())
print("Lowest Top-20 score:", df["score"].head(20).min())

print("\nReason codes in Top 20:")
print(df.head(20)["reason_code"].value_counts())


Baseline features: ['content_age_days', 'impressions_90d']

Required columns present:
content_age_days: True
impressions_90d: True

Prohibited fields used in baseline:
trend_direction: False
trend_pct: False
is_declining_label: False

Top 20 score range:
Highest score: 517715
Lowest Top-20 score: 181514

Reason codes in Top 20:
reason_code
stale_but_visible    20
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.